# 02 - Model Predictive Control

This notebook starts from the LQR idea and asks a practical question: what changes when the actuator and the predicted states must satisfy constraints?

The main message is visible in two closed-loop simulations:
- a short horizon needs a good terminal cost to behave like the infinite-horizon LQR;
- clipping LQR is not the same as solving a constrained optimal control problem.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import block_diag, solve_discrete_are
from scipy.optimize import Bounds, LinearConstraint, minimize

np.set_printoptions(precision=3, suppress=True)


## The Teaching Plant

We use the 2D discrete double integrator from the LQR notebook.

What are we comparing?
- position and velocity states;
- one scalar acceleration input;
- the same quadratic stage cost in every experiment.


In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.eye(2)
R = np.array([[1.0]])

nx = A.shape[0]
nu = B.shape[1]

P_inf = solve_discrete_are(A, B, Q, R)
K_inf = np.linalg.solve(R + B.T @ P_inf @ B, B.T @ P_inf @ A)

print("Infinite-horizon LQR gain K_inf =", K_inf)
print("DARE terminal matrix P_inf =\n", P_inf)


## A Short Horizon Has A Missing Tail

What are we comparing?
- receding-horizon controllers with the same short horizon;
- only the terminal cost changes.

What should students observe?
- no or weak terminal cost makes the controller more short-sighted;
- better terminal approximations move the closed loop toward LQR;
- using `P_inf` recovers infinite-horizon LQR in the unconstrained case.


In [ ]:
def riccati_update(P):
    K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
    return Q + A.T @ P @ (A - B @ K)


def riccati_iterates(number_of_tail_steps):
    P = np.zeros_like(Q)
    for _ in range(number_of_tail_steps):
        P = riccati_update(P)
    return P


def finite_horizon_lqr_gain(horizon, terminal_cost):
    P = terminal_cost.copy()
    K0 = None
    for _ in range(horizon):
        K0 = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)
        P = Q + A.T @ P @ (A - B @ K0)
    return K0


def simulate_linear_feedback(K, x_initial, steps):
    X = np.zeros((steps + 1, nx))
    U = np.zeros(steps)
    X[0] = x_initial
    for k in range(steps):
        U[k] = -float((K @ X[k]).item())
        X[k + 1] = A @ X[k] + B[:, 0] * U[k]
    return X, U


def simulate_receding_unconstrained(horizon, terminal_cost, x_initial, steps):
    X = np.zeros((steps + 1, nx))
    U = np.zeros(steps)
    X[0] = x_initial
    for k in range(steps):
        K0 = finite_horizon_lqr_gain(horizon, terminal_cost)
        U[k] = -float((K0 @ X[k]).item())
        X[k + 1] = A @ X[k] + B[:, 0] * U[k]
    return X, U


In [ ]:
terminal_horizon = 3
simulation_steps = 14
x_initial = np.array([5.0, 0.0])

terminal_costs = {
    "Vf = 0": np.zeros((2, 2)),
    "Vf = Q": Q,
    "Vf = P_iter_1": riccati_iterates(1),
    "Vf = P_iter_2": riccati_iterates(2),
    "Vf = P_iter_many": riccati_iterates(20),
    "Vf = P_inf": P_inf,
}

runs = {name: simulate_receding_unconstrained(terminal_horizon, Pf, x_initial, simulation_steps)
        for name, Pf in terminal_costs.items()}
runs["infinite-horizon LQR"] = simulate_linear_feedback(K_inf, x_initial, simulation_steps)

time = np.arange(simulation_steps + 1)
fig, axes = plt.subplots(3, 1, figsize=(8.0, 7.0), sharex=True)
for label, (X, U) in runs.items():
    linewidth = 2.4 if label in ["Vf = P_inf", "infinite-horizon LQR"] else 1.4
    linestyle = "--" if label == "infinite-horizon LQR" else "-"
    axes[0].plot(time, X[:, 0], label=label, linewidth=linewidth, linestyle=linestyle)
    axes[1].plot(time, X[:, 1], label=label, linewidth=linewidth, linestyle=linestyle)
    axes[2].step(time[:-1], U, where="post", label=label, linewidth=linewidth, linestyle=linestyle)

axes[0].set_ylabel("position")
axes[1].set_ylabel("velocity")
axes[2].set_ylabel("input")
axes[2].set_xlabel("time step")
for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="best", ncols=2)
fig.tight_layout()


### Interpretation

- The controller with `Vf = 0` sees only three future steps, so its first moves are visibly different.
- One or two Riccati tail steps already improve the behavior.
- `P_iter_many` is almost indistinguishable from `P_inf`.
- With `P_inf`, the unconstrained receding-horizon controller overlays the infinite-horizon LQR reference.


## Build The Finite-Horizon QP By Hand

Now constraints enter. We condense the prediction equations into

`X = Sx x0 + Su U`

where `U` is the whole planned input sequence.

Control message: MPC optimizes a sequence, not just the current clipped input.


In [ ]:
def prediction_matrices(horizon):
    Sx = np.zeros((horizon * nx, nx))
    Su = np.zeros((horizon * nx, horizon * nu))
    for i in range(horizon):
        Sx[i * nx:(i + 1) * nx, :] = np.linalg.matrix_power(A, i + 1)
        for j in range(i + 1):
            Su[i * nx:(i + 1) * nx, j * nu:(j + 1) * nu] = np.linalg.matrix_power(A, i - j) @ B
    return Sx, Su


def condensed_cost_matrices(horizon, terminal_cost):
    Sx, Su = prediction_matrices(horizon)
    Qbar = block_diag(*([Q] * (horizon - 1) + [Q + terminal_cost]))
    Rbar = block_diag(*([R] * horizon))
    H = Su.T @ Qbar @ Su + Rbar
    return Sx, Su, Qbar, H


def solve_constrained_qp(x_now, horizon, terminal_cost, u_limit, position_min=None):
    Sx, Su, Qbar, H = condensed_cost_matrices(horizon, terminal_cost)
    predicted_free = Sx @ x_now
    f = Su.T @ Qbar @ predicted_free
    bounds = Bounds(-u_limit * np.ones(horizon), u_limit * np.ones(horizon))
    constraints = []

    if position_min is not None:
        position_rows = Su[0::nx, :]
        position_free = predicted_free[0::nx]
        lower = position_min - position_free
        upper = np.full(horizon, np.inf)
        constraints.append(LinearConstraint(position_rows, lower, upper))

    result = minimize(
        lambda U: 0.5 * U @ H @ U + f @ U,
        np.zeros(horizon),
        jac=lambda U: H @ U + f,
        bounds=bounds,
        constraints=constraints,
        method="SLSQP",
        options={"ftol": 1e-9, "maxiter": 300},
    )
    return result


## Saturated LQR Versus Constrained MPC

What are we comparing?
- unconstrained LQR;
- the same LQR with immediate input clipping;
- MPC with both input bounds and a predicted position constraint.

The position constraint is `position >= 0`. The initial state has large positive velocity, so every controller first brakes. The difference appears later: saturated LQR respects the input bound but does not plan the future wall crossing.


In [ ]:
constrained_horizon = 12
constrained_steps = 30
u_limit = 0.5
position_min = 0.0
x_constrained_initial = np.array([0.0, 4.0])


def simulate_constrained_mpc(x_initial, steps):
    X = np.zeros((steps + 1, nx))
    U = np.zeros(steps)
    success = []
    X[0] = x_initial
    for k in range(steps):
        result = solve_constrained_qp(X[k], constrained_horizon, P_inf, u_limit, position_min=position_min)
        success.append(result.success)
        if not result.success:
            raise RuntimeError(f"MPC QP failed at step {k}: {result.message}")
        U[k] = result.x[0]
        X[k + 1] = A @ X[k] + B[:, 0] * U[k]
    return X, U, success


def simulate_saturated_lqr(x_initial, steps, u_limit):
    X = np.zeros((steps + 1, nx))
    U = np.zeros(steps)
    X[0] = x_initial
    for k in range(steps):
        raw = -float((K_inf @ X[k]).item())
        U[k] = np.clip(raw, -u_limit, u_limit)
        X[k + 1] = A @ X[k] + B[:, 0] * U[k]
    return X, U

X_lqr, U_lqr = simulate_linear_feedback(K_inf, x_constrained_initial, constrained_steps)
X_sat, U_sat = simulate_saturated_lqr(x_constrained_initial, constrained_steps, u_limit)
X_mpc, U_mpc, mpc_success = simulate_constrained_mpc(x_constrained_initial, constrained_steps)

constrained_runs = {
    "unconstrained LQR": (X_lqr, U_lqr),
    "saturated LQR": (X_sat, U_sat),
    "constrained MPC": (X_mpc, U_mpc),
}

print("MPC solver successes:", sum(mpc_success), "of", len(mpc_success))
print("minimum position:", {name: float(np.min(X[:, 0])) for name, (X, _) in constrained_runs.items()})
print("maximum |u|:", {name: float(np.max(np.abs(U))) for name, (_, U) in constrained_runs.items()})


In [ ]:
time = np.arange(constrained_steps + 1)
fig, axes = plt.subplots(4, 1, figsize=(8.0, 8.5), sharex=True)
for label, (X, U) in constrained_runs.items():
    linewidth = 2.2 if label == "constrained MPC" else 1.5
    axes[0].plot(time, X[:, 0], label=label, linewidth=linewidth)
    axes[1].plot(time, X[:, 1], label=label, linewidth=linewidth)
    axes[2].step(time[:-1], U, where="post", label=label, linewidth=linewidth)
    axes[3].plot(time, X[:, 0] - position_min, label=label, linewidth=linewidth)

axes[0].axhline(position_min, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(u_limit, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(-u_limit, color="k", linestyle="--", linewidth=0.9)
axes[3].axhline(0.0, color="k", linewidth=0.9)

axes[0].set_ylabel("position")
axes[1].set_ylabel("velocity")
axes[2].set_ylabel("input")
axes[3].set_ylabel("wall margin")
axes[3].set_xlabel("time step")
for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="best")
fig.tight_layout()


### Interpretation

- Unconstrained LQR asks for inputs outside the actuator limit.
- Saturated LQR clips those inputs, but it later crosses the wall because it never optimized the future trajectory under the wall constraint.
- MPC brakes and then plans the recovery sequence so the predicted positions stay feasible.
- Once constraints are active, the Riccati/LQR gain is no longer enough information.


## Inspect One MPC Plan

Before simulating, MPC predicts a whole sequence. Here we solve the first constrained problem and plot the planned position, velocity, and input.


In [ ]:
first_plan = solve_constrained_qp(x_constrained_initial, constrained_horizon, P_inf, u_limit, position_min=position_min)
planned_U = first_plan.x
Sx_plan, Su_plan = prediction_matrices(constrained_horizon)
planned_X = (Sx_plan @ x_constrained_initial + Su_plan @ planned_U).reshape(constrained_horizon, nx)
planned_time = np.arange(1, constrained_horizon + 1)

fig, axes = plt.subplots(3, 1, figsize=(8.0, 6.8), sharex=True)
axes[0].plot(planned_time, planned_X[:, 0], marker="o")
axes[0].axhline(position_min, color="k", linestyle="--", linewidth=0.9)
axes[1].plot(planned_time, planned_X[:, 1], marker="o")
axes[2].step(np.arange(constrained_horizon), planned_U, where="post", marker="o")
axes[2].axhline(u_limit, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(-u_limit, color="k", linestyle="--", linewidth=0.9)
axes[0].set_ylabel("planned position")
axes[1].set_ylabel("planned velocity")
axes[2].set_ylabel("planned input")
axes[2].set_xlabel("prediction step")
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.tight_layout()


## CasADi At The End

Now that the hand-built QP is visible, CasADi is useful as a formulation tool. It solves the same constrained MPC problem: double-integrator dynamics, input bounds, and `position >= 0` along the prediction horizon.


In [ ]:
import casadi as ca


def solve_constrained_qp_with_casadi(x_now):
    opti = ca.Opti()
    X = opti.variable(nx, constrained_horizon + 1)
    U = opti.variable(nu, constrained_horizon)
    cost = 0

    opti.subject_to(X[:, 0] == x_now)
    for k in range(constrained_horizon):
        xk = X[:, k]
        uk = U[:, k]
        opti.subject_to(X[:, k + 1] == A @ xk + B @ uk)
        opti.subject_to(opti.bounded(-u_limit, uk, u_limit))
        opti.subject_to(X[0, k + 1] >= position_min)
        cost += ca.mtimes([xk.T, Q, xk]) + ca.mtimes([uk.T, R, uk])
    terminal = X[:, constrained_horizon]
    cost += ca.mtimes([terminal.T, P_inf, terminal])

    opti.minimize(cost)
    opti.solver("ipopt", {"print_time": False}, {"print_level": 0})
    sol = opti.solve()
    return np.array(sol.value(U[:, 0])).reshape(-1)

casadi_first_u = solve_constrained_qp_with_casadi(x_constrained_initial)
print("first input from hand QP  =", U_mpc[0])
print("first input from CasADi QP =", casadi_first_u[0])
